<a href="https://colab.research.google.com/github/Jahdejosefina/projecto_ragTP2/blob/main/00_corpus_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive to Server...' from Ctrl+Shift+P, then re-run this cell)
Mounted at /content/drive
artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
Colab bootstrap OK. Working directory: /content/projecto_rag


# Etapa 0 · El corpus como decisión

**Este notebook es el ejemplo resuelto.** Audita uno de los corpus del menú --
**python-docs-es**, la traducción al español de la documentación de Python -- de la forma en
que la Etapa 0 te pide que audites el tuyo. Nada de esto construye el pipeline; mide qué *es*
el corpus antes de que decidas qué hacer con él.

Por qué molestarse: haga lo que haga tu pipeline de recuperación, no puede recuperar algo que
los documentos no contienen. El corpus es el techo. Así que antes de elegir tamaños de chunk o
rerankers, respondé las preguntas que responde esta auditoría -- con números, no con impresiones:

1. **¿Está la respuesta ahí?** (¿y en qué idioma?)
2. **¿El texto está limpio?** ¿Qué dejaría pasar un loader ingenuo sin ningún error?
3. **¿Tiene estructura** por la que se pueda cortar?
4. **¿Cada documento tiene un identificador estable** -- uno que sobreviva a una nueva descarga?
5. **¿Hay metadatos** -- una fecha, una versión, un estado -- y son correctos?
6. **¿Cambia,** y cómo te enterarías?
7. **¿Se puede armar un conjunto de evaluación** sobre él?
8. Y una que no es una propiedad sino un hecho: **¿cuán parecidos son los documentos entre
   sí?** Los vecinos confundibles son lo que hace difícil la recuperación.

Si elegiste python-docs-es, esta auditoría es **donde tu Etapa 0 empieza, no donde termina**:
deja abierta a propósito la decisión que más importa -- por qué ruta entrás al corpus -- y
varios de sus números merecen una segunda mirada.

In [2]:
import environment  # noqa: F401  (efectos secundarios: guarda de JAX, caché de HF, bundle TLS, UTF-8)

environment.summary()

from console import StageRun, clip, print_table, verdict

SMOKE_TEST = True  # True: unas pocas páginas, minutos en una laptop. False: el subconjunto entero.
SEED = 1337
ART = environment.artifacts_dir(SMOKE_TEST)
PROFILE = "smoke" if SMOKE_TEST else "full"
run = StageRun.start("00_corpus_audit", ART, profile=PROFILE, config={"seed": SEED})

ENVIRONMENT
os             : Linux 6.6.122+
python         : 3.13.15
interpreter    : /usr/bin/python3
colab          : True
logical_cores  : 2
torch          : 2.11.0+cu130
cuda_available : True
gpu            : Tesla T4
gpu_memory_gb  : 15.64
ram_total_gb   : 13.6
ram_free_gb    : 11.9
HF_HOME        : /content/drive/MyDrive/projecto_rag/hf_cache
jax_guard      : True
ca_bundle      : not present (normal)
artifacts      : -> /content/drive/MyDrive/projecto_rag/artifacts (Drive)
== 00_corpus_audit · profile smoke · report -> /content/projecto_rag/artifacts/smoke/run_report.json


In [5]:
import store
import os

# 1. El repositorio de las leyes argentinas sugerido en corpora.md
LEGAL_REPO = "https://github.com/legalize-dev/legalize-ar"

corpora = environment.corpora_dir()
legal_root = corpora / "legalize-ar"

# Descargamos el repositorio (rama main).
# El enunciado pide fijar un commit; para la auditoría inicial traeremos la última versión.
store.fetch_git(LEGAL_REPO, legal_root, branch="main", commit="3d2450b")

# Verificamos que se haya descargado y contamos los archivos
md_files = list(legal_root.rglob("*.md"))
print(f"Repositorio clonado en: {legal_root}")
print(f"Cantidad de archivos Markdown (normas) encontrados: {len(md_files)}")

reusing legalize-ar @ 3d2450b5c09a
Repositorio clonado en: /content/projecto_rag/corpora/legalize-ar
Cantidad de archivos Markdown (normas) encontrados: 31327


### El tema elegido: Defensa del consumidor (Ley 24.240 y modificatorias)

In [7]:
palabras_clave = ["defensa del consumidor", "24.240", "ley 24240"]

subconjunto = []
for path in md_files:
    # Leemos el texto en minúsculas para facilitar la búsqueda
    texto = path.read_text(encoding="utf-8").lower()

    # Si alguna de las palabras clave está en el texto de la ley, la guardamos
    if any(keyword in texto for keyword in palabras_clave):
        subconjunto.append(path)

print(f"Archivos encontrados para Defensa del Consumidor: {len(subconjunto)}")

# Imprimimos los nombres de los primeros 10 archivos para verificar qué agarró
print("\nPrimeros 10 archivos del subconjunto:")
for path in subconjunto[:10]:
    print(" -", path.name)

Archivos encontrados para Defensa del Consumidor: 170

Primeros 10 archivos del subconjunto:
 - DNU-271-2000.md
 - DEC-946-1989.md
 - LEY-27250.md
 - DNU-13-2015.md
 - DNU-1283-2003.md
 - DEC-991-2020.md
 - DEC-528-2011.md
 - DEC-276-1998.md
 - DNU-2156-2002.md
 - DEC-826-2011.md


## ¿Hace falta siquiera la recuperación?

La línea de base honesta para cualquier sistema de RAG es: *si entra, pegalo*. Entonces: ¿qué
tan grande es este corpus en los tokens del modelo que lo va a leer -- no en caracteres, no en
palabras, y no en los tokens del embedder (cada modelo divide el mismo texto de forma distinta)?

In [8]:
import re
import llm

GENERATOR = "qwen3:8b"
NUM_CTX = 8192

fechas_malas = 0
encabezados_encontrados = set()
total_chars = 0
texto_completo_muestra = ""

for path in subconjunto:
    texto = path.read_text(encoding="utf-8")
    total_chars += len(texto)
    # Guardamos una muestra para que el LLM calcule la proporción de tokens por caracter
    if len(texto_completo_muestra) < 400_000:
        texto_completo_muestra += texto + "\n"

    # 1. Chequear metadatos engañosos (Pregunta 5 de la auditoría)
    if 'last_updated: "1900-01-01"' in texto:
        fechas_malas += 1

    # 2. Buscar variaciones tipográficas de "ARTICULO" (Pregunta 2 de la auditoría)
    # Busca cosas como "ARTICULO 1º", "Artículo 1º.-", "###### ARTICULO 1"
    matches = re.findall(r'(?i)(#*\s*art[íi]culo\s+\d+[º°]?\s*[—\-\.]*)', texto)
    # Guardamos solo un par de ejemplos para no saturar la pantalla
    for m in matches[:2]:
        encabezados_encontrados.add(m.strip())

# 3. Estimar tamaño en tokens (Pregunta de la línea de base)
# Requiere que Ollama esté corriendo. Si falla, el ratio suele ser ~0.3 tokens/char.
try:
    tokens_per_char = llm.count_tokens(texto_completo_muestra, GENERATOR) / max(len(texto_completo_muestra), 1)
except Exception:
    print("(Ollama no está levantado, usando estimación aproximada de 0.3 tokens por caracter)")
    tokens_per_char = 0.30

est_tokens = int(total_chars * tokens_per_char)

print(f"--- RESULTADOS DE LA AUDITORÍA DE DEFENSA DEL CONSUMIDOR ---")
print(f"Archivos analizados: {len(subconjunto)}")
print(f"Leyes con fecha 'last_updated' falsa (1900-01-01): {fechas_malas}")
print(f"Ejemplos de cómo escriben 'Artículo': {list(encabezados_encontrados)[:5]}")
print(f"\nCaracteres totales del subconjunto: {total_chars:,}")
print(f"Tokens estimados para el LLM: ~{est_tokens:,} tokens")
print(f"Equivale a {est_tokens / NUM_CTX:.1f} ventanas de {NUM_CTX} tokens.")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (450168 > 131072). Running this sequence through the model will result in indexing errors


--- RESULTADOS DE LA AUDITORÍA DE DEFENSA DEL CONSUMIDOR ---
Archivos analizados: 170
Leyes con fecha 'last_updated' falsa (1900-01-01): 159
Ejemplos de cómo escriben 'Artículo': ['Artículo 14', 'ARTICULO 113.-', 'ARTICULO 155', 'ARTÍCULO 1º —', 'ARTICULO 13.']

Caracteres totales del subconjunto: 10,835,049
Tokens estimados para el LLM: ~3,041,374 tokens
Equivale a 371.3 ventanas de 8192 tokens.


In [9]:
contenido = """# Ficha del Corpus: Legislación Argentina (Defensa del Consumidor)

- **Fuente y versión fijada:** Repositorio `legalize-ar` (commit `3d2450b`).
- **Licencia y atribución:** CC BY 4.0 (SAIJ / Ministerio de Justicia).
- **Regla del subconjunto:** 170 normas que contienen las palabras clave "defensa del consumidor", "24.240" o "ley 24240".
- **Defectos conocidos (Datos sucios):**
  - Los metadatos de fechas mienten: 159 de 170 archivos reportan `last_updated: "1900-01-01"`.
  - Variabilidad tipográfica extrema en los encabezados (ej. `ARTICULO 113.-`, `ARTÍCULO 1º —`, `Artículo 14`), lo que exige expresiones regulares tolerantes para el chunking.
- **Tamaño:** ~3.040.000 tokens del generador qwen3:8b (equivale a más de 371 ventanas de 8192 tokens). La recuperación es estrictamente necesaria.
- **Qué pone a prueba bien el corpus:** Estructura jerárquica clara (artículos) y actualizaciones de conocimiento en el tiempo (reformas sobre la misma ley).
"""

with open("corpus_card.md", "w", encoding="utf-8") as f:
    f.write(contenido)

print("¡Archivo corpus_card.md creado y guardado con éxito!")

¡Archivo corpus_card.md creado y guardado con éxito!


In [11]:
import statistics

# 1. Distribución de largos en tokens
largos_tokens = [len(path.read_text(encoding="utf-8")) * tokens_per_char for path in subconjunto]
print("--- 1. DISTRIBUCIÓN DE LARGOS ---")
print(f"Largo promedio por documento: {statistics.mean(largos_tokens):.0f} tokens")
print(f"Documento más largo: {max(largos_tokens):.0f} tokens")
print(f"Documento más corto: {min(largos_tokens):.0f} tokens")

# 2. Casi-duplicados (búsqueda rápida de nombres o tamaños idénticos)
tamaños = [len(path.read_text(encoding="utf-8")) for path in subconjunto]
duplicados_aparentes = len(tamaños) - len(set(tamaños))
print(f"\n--- 2. CASI-DUPLICADOS ---")
print(f"Documentos con exactamente la misma cantidad de caracteres (posibles duplicados): {duplicados_aparentes}")

# 3. Chequeo a libro cerrado (5 preguntas sin recuperación)
print("\n--- 3. CHEQUEO A LIBRO CERRADO ---")
preguntas_libro_cerrado = [
    "¿Qué dice el artículo 4 de la Ley 24.240 de Defensa del Consumidor sobre la información?",
    "¿Cuáles son las sanciones principales previstas en la Ley de Defensa del Consumidor en Argentina?",
    "¿Qué es el daño punitivo en el derecho del consumidor argentino?",
    "¿Qué plazo hay para reclamar por vicios redhibitorios según la Ley 24.240?",
    "¿Es válido un contrato de consumo que renuncia a los derechos del consumidor en Argentina?"
]

llm.setup(GENERATOR, num_ctx=NUM_CTX)
if llm.server_alive() and llm.model_digest(GENERATOR):
    for q in preguntas_libro_cerrado:
        r = llm.ask(q, system="Respondé en una oración.", model=GENERATOR, num_ctx=NUM_CTX, num_predict=160, cache=False)
        print(f"Pregunta: {q}\nRespuesta del modelo: {r.text[:200]}\n")
else:
    print("El servidor de Ollama no está respondiendo para hacer el test a libro cerrado.")

--- 1. DISTRIBUCIÓN DE LARGOS ---
Largo promedio por documento: 17890 tokens
Documento más largo: 342957 tokens
Documento más corto: 344 tokens

--- 2. CASI-DUPLICADOS ---
Documentos con exactamente la misma cantidad de caracteres (posibles duplicados): 1

--- 3. CHEQUEO A LIBRO CERRADO ---
installing Ollama 0.34.4 (about a minute) ...
Ollama server up on http://127.0.0.1:11434 (log: /tmp/ollama_serve.log)
pulling qwen3:8b ...
     0%  0.01/5.23 GB
    10%  0.53/5.23 GB
    20%  1.05/5.23 GB
    30%  1.57/5.23 GB
    40%  2.10/5.23 GB
    50%  2.61/5.23 GB
    60%  3.16/5.23 GB
    70%  3.66/5.23 GB
    80%  4.19/5.23 GB
    90%  4.72/5.23 GB
   model        qwen3:8b
   digest       500a1f067a9f
   server       0.34.4
   num_ctx      8192
   thinking     {'honors_think_false': True, 'leaks_think_tag': False, 'thinking_only': False, 'supports_thinking': True, 'answer': 'Paris'}
   key_order    {'order': 'declaration', 'reply': '{"zeta": "alpha", "alfa": "omega"}', 'error': None}
   cali

## La auditoría, en una tabla

Las ocho preguntas del principio, respondidas para este corpus con los números de arriba. Tu
Etapa 0 completa la misma tabla para el tuyo -- y agrega una columna que esta deja en blanco a
propósito: **qué etapa posterior paga por cada "no".**

In [13]:
properties = [
    ("1 answer in it", "Sí, en español (leyes nacionales argentinas)."),
    ("2 clean text", f"NO. Variabilidad tipográfica extrema en 'Artículo' (ej. {list(encabezados_encontrados)[:2]})."),
    ("3 structure", "Sí, jerarquía clara: artículos, incisos."),
    ("4 stable ids", "Sí, número de ley y número de artículo."),
    ("5 metadata", f"NO. {fechas_malas} de {len(subconjunto)} archivos tienen last_updated '1900-01-01' y status miente."),
    ("6 changes", "Sí, mediante reformas rastreables en commits de git."),
    ("7 eval set", "Sí, ideal para medir vigencia temporal y actualizaciones."),
    ("8 similarity", "Alta. Versiones consecutivas de una ley reformada son casi idénticas."),
]

print_table("legalize-ar (Defensa del Consumidor): la auditoría", ["medido"], [(p, {"medido": m}) for p, m in properties], smoke=SMOKE_TEST)

# Guardamos los resultados reales de tu subconjunto para que la cátedra pueda verlos
store.write_json(ART / "stage0_audit.json", {
    "profile": PROFILE,
    "corpus": "legalize-ar",
    "tema": "Defensa del Consumidor",
    "archivos": len(subconjunto),
    "datos_sucios": {
        "fechas_malas_1900": fechas_malas,
        "variaciones_articulo": list(encabezados_encontrados)[:5]
    },
    "size": {
        "characters": total_chars,
        "tokens_per_char": tokens_per_char,
        "est_tokens": est_tokens
    }
})

run.note("est_tokens", est_tokens)
run.finish()


legalize-ar (Defensa del Consumidor): la auditoría
[SMOKE_TEST -- these numbers mean nothing; they only prove the code runs]
------------------------------------------------------------------------------------------------------------
               | medido                                                                                     
------------------------------------------------------------------------------------------------------------
1 answer in it | Sí, en español (leyes nacionales argentinas).                                              
2 clean text   | NO. Variabilidad tipográfica extrema en 'Artículo' (ej. ['Artículo 14', 'ARTICULO 113.-']).
3 structure    | Sí, jerarquía clara: artículos, incisos.                                                   
4 stable ids   | Sí, número de ley y número de artículo.                                                    
5 metadata     | NO. 159 de 170 archivos tienen last_updated '1900-01-01' y status miente.                  
6 

{'stage': '00_corpus_audit',
 'profile': 'smoke',
 'config': {'seed': 1337},
 'status': 'ok',
 'started_at': '2026-10-18 00:51:41',
 'notes': {'est_tokens': 3041374},
 'llm': {'calls': 13,
  'cache_hits': 0,
  'errors': 0,
  'prompt_tokens': 1129,
  'output_tokens': 400,
  'silent_truncations': 0,
  'server_refusals': 0,
  'output_truncations': 5,
  'think_leaks': 0,
  'reloads': 1},
 'wall_s': 3118.8,
 'peak_vram_gb': 0.0,
 'finished_at': '2026-10-18 01:43:40'}

## Para tu informe

- **Qué ruta de entrada al corpus, y por qué.** Esta auditoría midió las dos y no eligió
  ninguna. La ruta `.po` tiene las marcas de calidad de los propios traductores y un historial de
  git; la ruta HTML tiene firmas, notas de versión e ids estables, y esconde la división
  fuzzy/sin traducir detrás de texto en inglés. Tomes la que tomes, nombrá lo que te cuesta más
  adelante.
- **¿Por qué el directorio con más inglés tiene tanto inglés?** Las marcas de la compilación y la
  comparación de versiones de arriba apuntan a una causa que no tiene nada que ver con el
  esfuerzo de los traductores. ¿Qué significa para un usuario que pregunta sobre esas páginas en
  cada versión?
- **¿Qué propiedades de las siete no cumple tu corpus -- y cuánto te cuesta la octava, la
  similitud -- y en qué etapa vas a pagar por cada una?** Una oración por cada "no".
- **¿Entra en una ventana?** Ahora tenés el número. Si entrara, ¿para qué construir
  recuperación?
- **¿Cuánto ya leyó el generador?** Mirá las respuestas a libro cerrado. ¿En cuáles confiaría un
  usuario -- y cuáles están mal de una forma que solo los documentos podrían corregir?
- **¿Qué hace que dos de tus documentos se puedan confundir,** y es una propiedad que querés
  conservar en tu evaluación o eliminar con ingeniería?

Siguiente: **Etapa 1** (`01_documents`) -- convertir la fuente en documentos: un registro por
documento, ids estables, secciones, metadatos que verificaste contra el texto.